<a href="https://colab.research.google.com/github/YOUSSEF-BT/BIGDATA_LAB_Spark_SQL_FIFA/blob/main/TP_Spark_SQL_FIFA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**installation pyspark**

In [60]:
!pip install -q pyspark

In [61]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TempView Example") \
  .master("local[*]") \
  .getOrCreate()

df = spark.read.option("header", True).option("inferSchema", True) \
  .csv("/content/fifaworldcup.csv")

df.createOrReplaceTempView("Matches")

In [62]:
result = spark.sql("SELECT * FROM Matches")
result.show(10)

+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|      date|home_team|away_team|home_score|away_score|tournament|   city| country|neutral|
+----------+---------+---------+----------+----------+----------+-------+--------+-------+
|1872-11-30| Scotland|  England|         0|         0|  Friendly|Glasgow|Scotland|  false|
|1873-03-08|  England| Scotland|         4|         2|  Friendly| London| England|  false|
|1874-03-07| Scotland|  England|         2|         1|  Friendly|Glasgow|Scotland|  false|
|1875-03-06|  England| Scotland|         2|         2|  Friendly| London| England|  false|
|1876-03-04| Scotland|  England|         3|         0|  Friendly|Glasgow|Scotland|  false|
|1876-03-25| Scotland|    Wales|         4|         0|  Friendly|Glasgow|Scotland|  false|
|1877-03-03|  England| Scotland|         1|         3|  Friendly| London| England|  false|
|1877-03-05|    Wales| Scotland|         0|         2|  Friendly|Wrexham|   Wales|  false|

In [63]:
df.printSchema()

root
 |-- date: date (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_score: string (nullable = true)
 |-- away_score: string (nullable = true)
 |-- tournament: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- neutral: boolean (nullable = true)



In [64]:
df.show()
df.select("date","home_team","away_team","home_score","away_score","tournament").show()

+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|      date|       home_team|away_team|home_score|away_score|tournament|     city| country|neutral|
+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|1872-11-30|        Scotland|  England|         0|         0|  Friendly|  Glasgow|Scotland|  false|
|1873-03-08|         England| Scotland|         4|         2|  Friendly|   London| England|  false|
|1874-03-07|        Scotland|  England|         2|         1|  Friendly|  Glasgow|Scotland|  false|
|1875-03-06|         England| Scotland|         2|         2|  Friendly|   London| England|  false|
|1876-03-04|        Scotland|  England|         3|         0|  Friendly|  Glasgow|Scotland|  false|
|1876-03-25|        Scotland|    Wales|         4|         0|  Friendly|  Glasgow|Scotland|  false|
|1877-03-03|         England| Scotland|         1|         3|  Friendly|   London| England|  false|


**Maroc (10 derniers matchs)**

In [65]:
spark.sql("""
SELECT date, home_team, away_team, home_score, away_score, tournament
FROM matches
WHERE (home_team='Morocco' OR away_team='Morocco')
  AND home_score IS NOT NULL AND away_score IS NOT NULL
ORDER BY date DESC
""").show()

+----------+-------------+------------+----------+----------+--------------------+
|      date|    home_team|   away_team|home_score|away_score|          tournament|
+----------+-------------+------------+----------+----------+--------------------+
|2022-12-01|       Canada|     Morocco|        NA|        NA|      FIFA World Cup|
|2022-11-27|      Belgium|     Morocco|        NA|        NA|      FIFA World Cup|
|2022-11-23|      Morocco|     Croatia|        NA|        NA|      FIFA World Cup|
|2022-09-27|     Paraguay|     Morocco|         0|         0|            Friendly|
|2022-09-23|      Morocco|       Chile|         2|         0|            Friendly|
|2022-06-13|      Morocco|     Liberia|         2|         0|African Cup of Na...|
|2022-06-09|      Morocco|South Africa|         2|         1|African Cup of Na...|
|2022-06-01|United States|     Morocco|         3|         0|            Friendly|
|2022-03-29|      Morocco|    DR Congo|         4|         1|FIFA World Cup qu...|
|202

**Stats Maroc (matchs / victoires / nuls / défaites)**

In [66]:
spark.sql("""
WITH maroc AS (
  SELECT
    CASE WHEN home_team='Morocco' THEN home_score ELSE away_score END bm,
    CASE WHEN home_team='Morocco' THEN away_score ELSE home_score END ba
  FROM matches
  WHERE home_team='Morocco' OR away_team='Morocco'
)
SELECT
  COUNT(*) matchs,
  SUM(CASE WHEN bm>ba THEN 1 ELSE 0 END) victoires,
  SUM(CASE WHEN bm=ba THEN 1 ELSE 0 END) nuls,
  SUM(CASE WHEN bm<ba THEN 1 ELSE 0 END) defaites
FROM maroc
WHERE bm IS NOT NULL AND ba IS NOT NULL
""").show()

+------+---------+----+--------+
|matchs|victoires|nuls|defaites|
+------+---------+----+--------+
|   575|      271| 167|     137|
+------+---------+----+--------+



**Cellule 12 Buts pour/contre du Maroc**

In [67]:
spark.sql("""
WITH maroc AS (
  SELECT
    CASE WHEN home_team='Morocco' THEN try_cast(home_score AS INT)
         ELSE try_cast(away_score AS INT) END AS bp,
    CASE WHEN home_team='Morocco' THEN try_cast(away_score AS INT)
         ELSE try_cast(home_score AS INT) END AS bc
  FROM matches
  WHERE home_team='Morocco' OR away_team='Morocco'
)
SELECT
  SUM(bp) AS buts_pour,
  SUM(bc) AS buts_contre,
  SUM(bp) - SUM(bc) AS diff
FROM maroc
WHERE bp IS NOT NULL AND bc IS NOT NULL
""").show()

+---------+-----------+----+
|buts_pour|buts_contre|diff|
+---------+-----------+----+
|      829|        481| 348|
+---------+-----------+----+



**Cellule 13 — Préparer les types**

In [68]:
from pyspark.sql import functions as F

df2 = (df
       .withColumn("date", F.to_date("date"))
       .withColumn("home_score", F.col("home_score").cast("int"))
       .withColumn("away_score", F.col("away_score").cast("int"))
       .withColumn("neutral", F.col("neutral").cast("boolean"))
      )

df2.createOrReplaceTempView("Matches")
df2.createOrReplaceTempView("matches")

**Cellule 14 — Vues utiles pour les questions par équipe**

In [69]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW team_matches AS
SELECT
  date,
  year(date) AS year,
  CAST(floor(year(date)/10)*10 AS INT) AS decade,
  tournament,
  country,
  city,
  neutral,
  home_team AS team,
  away_team AS opponent,
  home_score AS goals_for,
  away_score AS goals_against,
  true AS is_home
FROM matches
UNION ALL
SELECT
  date,
  year(date) AS year,
  CAST(floor(year(date)/10)*10 AS INT) AS decade,
  tournament,
  country,
  city,
  neutral,
  away_team AS team,
  home_team AS opponent,
  away_score AS goals_for,
  home_score AS goals_against,
  false AS is_home
FROM matches
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW team_results AS
SELECT *,
  CASE
    WHEN goals_for > goals_against THEN 'W'
    WHEN goals_for < goals_against THEN 'L'
    ELSE 'D'
  END AS result
FROM team_matches
""")

DataFrame[]

**Cellule 15 — Q1**

In [70]:
spark.sql("""
SELECT COUNT(*) AS nb_matches
FROM matches
""").show()

+----------+
|nb_matches|
+----------+
|     44152|
+----------+



**Cellule 16 — Q2**

In [71]:
spark.sql("""
SELECT MIN(year(date)) AS first_year, MAX(year(date)) AS last_year
FROM matches
""").show()

+----------+---------+
|first_year|last_year|
+----------+---------+
|      1872|     2022|
+----------+---------+



**Cellule 17 — Q3**

In [72]:
spark.sql("""
SELECT tournament, COUNT(*) AS n
FROM matches
GROUP BY tournament
ORDER BY n DESC
LIMIT 10
""").show()

+--------------------+-----+
|          tournament|    n|
+--------------------+-----+
|            Friendly|17461|
|FIFA World Cup qu...| 7774|
|UEFA Euro qualifi...| 2593|
|African Cup of Na...| 1932|
|      FIFA World Cup|  948|
|        Copa América|  841|
|AFC Asian Cup qua...|  764|
|African Cup of Na...|  742|
|          CECAFA Cup|  620|
|CFU Caribbean Cup...|  606|
+--------------------+-----+



**Cellule 18 — Q4**

In [73]:
spark.sql("""
SELECT COUNT(*) AS nb_neutral
FROM matches
WHERE neutral = true
""").show()

+----------+
|nb_neutral|
+----------+
|     10996|
+----------+



**Cellule 19 — Q5**

In [74]:
spark.sql("""
SELECT country, COUNT(*) AS n
FROM matches
GROUP BY country
ORDER BY n DESC
LIMIT 10
""").show()

+--------------------+----+
|             country|   n|
+--------------------+----+
|       United States|1259|
|              France| 830|
|            Malaysia| 752|
|             England| 722|
|              Sweden| 659|
|               Qatar| 618|
|             Germany| 608|
|              Brazil| 569|
|               Spain| 568|
|United Arab Emirates| 548|
+--------------------+----+



**Cellule 20 — Q6**

In [75]:
from pyspark.sql import functions as F

df_clean = (df
    .withColumn("date", F.to_date("date"))
    .withColumn("home_score", F.when(F.col("home_score")=="NA", None).otherwise(F.col("home_score")).cast("int"))
    .withColumn("away_score", F.when(F.col("away_score")=="NA", None).otherwise(F.col("away_score")).cast("int"))
    .withColumn("neutral", F.col("neutral").cast("boolean"))
)

df_clean.createOrReplaceTempView("matches")
df_clean.createOrReplaceTempView("Matches")

In [76]:
spark.sql("""
SELECT COUNT(*) AS nb_draws
FROM matches
WHERE home_score IS NOT NULL
  AND away_score IS NOT NULL
  AND home_score = away_score
""").show()

+--------+
|nb_draws|
+--------+
|   10165|
+--------+



**Cellule 21 — Q7**

In [77]:
spark.sql("""
SELECT date, home_team, away_team, home_score, away_score, tournament, country, city, neutral
FROM matches
WHERE (home_score + away_score) > 6
ORDER BY date DESC
""").show()

+----------+--------------------+--------------------+----------+----------+--------------------+-------------------+-------------------+-------+
|      date|           home_team|           away_team|home_score|away_score|          tournament|            country|               city|neutral|
+----------+--------------------+--------------------+----------+----------+--------------------+-------------------+-------------------+-------+
|2022-11-05|              Brunei|         Timor-Leste|         6|         2|AFF Championship ...|             Brunei|Bandar Seri Begawan|  false|
|2022-06-14|           Indonesia|               Nepal|         7|         0|AFC Asian Cup qua...|             Kuwait|        Kuwait City|   true|
|2022-06-14|             Myanmar|           Singapore|         2|         6|AFC Asian Cup qua...|         Kyrgyzstan|            Bishkek|   true|
|2022-06-14|             Germany|               Italy|         5|         2| UEFA Nations League|            Germany|    Mön

**Cellule 22 — Q8**

In [78]:
spark.sql("""
SELECT team, COUNT(*) AS matches_played
FROM team_matches
GROUP BY team
ORDER BY matches_played DESC
""").show()

+-----------+--------------+
|       team|matches_played|
+-----------+--------------+
|     Sweden|          1052|
|    England|          1047|
|     Brazil|          1019|
|  Argentina|          1014|
|    Germany|           986|
|    Hungary|           964|
|     Mexico|           926|
|    Uruguay|           919|
|South Korea|           904|
|     France|           875|
|     Poland|           851|
|      Italy|           837|
|Switzerland|           834|
|     Norway|           832|
|    Denmark|           832|
|    Austria|           820|
|Netherlands|           820|
|   Scotland|           814|
|    Belgium|           804|
|      Chile|           793|
+-----------+--------------+
only showing top 20 rows


**Cellule 23 — Q9**

In [79]:
spark.sql("""
SELECT team, SUM(goals_for) AS goals_scored
FROM team_matches
GROUP BY team
ORDER BY goals_scored DESC
LIMIT 10
""").show()

+-----------+------------+
|       team|goals_scored|
+-----------+------------+
|    England|        2282|
|     Brazil|        2228|
|    Germany|        2205|
|     Sweden|        2060|
|    Hungary|        1948|
|  Argentina|        1880|
|Netherlands|        1694|
|     Mexico|        1592|
|South Korea|        1576|
|     France|        1560|
+-----------+------------+



**Cellule 24 — Q10**

In [80]:
spark.sql("""
SELECT
  CAST(floor(year(date)/10)*10 AS INT) AS decade,
  AVG(home_score + away_score) AS avg_goals_per_match
FROM matches
GROUP BY CAST(floor(year(date)/10)*10 AS INT)
ORDER BY decade
""").show()

+------+-------------------+
|decade|avg_goals_per_match|
+------+-------------------+
|  1870|  4.538461538461538|
|  1880|  5.581818181818182|
|  1890| 5.1525423728813555|
|  1900|  4.169354838709677|
|  1910|  4.007067137809187|
|  1920| 3.8367626886145403|
|  1930|   4.25940594059406|
|  1940|  4.294776119402985|
|  1950|  3.980964467005076|
|  1960| 3.4683274021352313|
|  1970|  2.955379908210097|
|  1980| 2.4950990615224193|
|  1990|  2.730442692540934|
|  2000|  2.792825302483549|
|  2010|   2.69994863893169|
|  2020|  2.600644864117918|
+------+-------------------+



**Cellule 25 — Q11**

In [93]:
spark.sql("""
SELECT year(date) AS year, tournament, COUNT(*) AS n
FROM matches
GROUP BY year(date), tournament
ORDER BY year DESC, n DESC
""").show()

+----+--------------------+---+
|year|          tournament|  n|
+----+--------------------+---+
|2022|            Friendly|281|
|2022| UEFA Nations League|160|
|2022|FIFA World Cup qu...| 99|
|2022|CONCACAF Nations ...| 67|
|2022|African Cup of Na...| 54|
|2022|African Cup of Na...| 52|
|2022|      FIFA World Cup| 48|
|2022|AFC Asian Cup qua...| 36|
|2022|MSG Prime Ministe...|  6|
|2022|          King's Cup|  4|
|2022|          Navruz Cup|  4|
|2022|Jordan Internatio...|  4|
|2022|CONIFA Africa Foo...|  4|
|2022|           Kirin Cup|  4|
|2022|CONIFA South Amer...|  3|
|2022| Kirin Challenge Cup|  2|
|2022|AFF Championship ...|  2|
|2022|          Baltic Cup|  2|
|2022|CONMEBOL–UEFA Cup...|  1|
|2021|FIFA World Cup qu...|607|
+----+--------------------+---+
only showing top 20 rows


**Cellule 26 — Q12**

In [82]:
spark.sql("""
SELECT home_team AS team, COUNT(*) AS home_wins
FROM matches
WHERE home_score > away_score
GROUP BY home_team
ORDER BY home_wins DESC
""").show()

+-------------+---------+
|         team|home_wins|
+-------------+---------+
|       Brazil|      423|
|    Argentina|      373|
|      Germany|      327|
|      England|      324|
|       Mexico|      320|
|  South Korea|      296|
|       Sweden|      295|
|        Italy|      291|
|       France|      290|
|      Hungary|      265|
|        Egypt|      260|
|        Spain|      256|
|  Netherlands|      252|
|United States|      243|
|      Denmark|      230|
|      Belgium|      229|
|     Scotland|      224|
| Saudi Arabia|      222|
|      Austria|      220|
|        Chile|      213|
+-------------+---------+
only showing top 20 rows


**Cellule 27 — Q13**

In [83]:
spark.sql("""
SELECT
  team,
  SUM(CASE WHEN result='W' THEN 1 ELSE 0 END) AS wins,
  SUM(CASE WHEN result='D' THEN 1 ELSE 0 END) AS draws,
  SUM(CASE WHEN result='L' THEN 1 ELSE 0 END) AS losses
FROM team_results
GROUP BY team
ORDER BY wins DESC
""").show()

+-----------+----+-----+------+
|       team|wins|draws|losses|
+-----------+----+-----+------+
|     Brazil| 651|  208|   160|
|    England| 594|  253|   200|
|    Germany| 573|  208|   205|
|  Argentina| 547|  254|   213|
|     Sweden| 517|  227|   308|
|South Korea| 477|  235|   192|
|     Mexico| 467|  216|   243|
|    Hungary| 452|  211|   301|
|      Italy| 445|  235|   157|
|     France| 436|  190|   249|
|      Spain| 424|  173|   134|
|Netherlands| 419|  188|   213|
|    Uruguay| 399|  226|   294|
|   Scotland| 386|  176|   252|
|    Denmark| 379|  176|   277|
|     Russia| 366|  187|   163|
|     Poland| 365|  218|   268|
|    Belgium| 354|  173|   277|
|     Zambia| 347|  201|   218|
|    Austria| 340|  177|   303|
+-----------+----+-----+------+
only showing top 20 rows


**Cellule 28 — Q14**

In [84]:
spark.sql("""
SELECT neutral, AVG(home_score + away_score) AS avg_total_goals
FROM matches
GROUP BY neutral
ORDER BY neutral
""").show()

+-------+------------------+
|neutral|   avg_total_goals|
+-------+------------------+
|  false|2.8861339848580823|
|   true|3.0150671171582504|
+-------+------------------+



**Cellule 29 — Q15**

In [85]:
spark.sql("""
SELECT
  date, home_team, away_team, home_score, away_score, tournament, country,
  ABS(home_score - away_score) AS goal_diff
FROM matches
ORDER BY goal_diff DESC, (home_score + away_score) DESC
LIMIT 5
""").show()

+----------+---------+--------------+----------+----------+--------------------+----------------+---------+
|      date|home_team|     away_team|home_score|away_score|          tournament|         country|goal_diff|
+----------+---------+--------------+----------+----------+--------------------+----------------+---------+
|2001-04-11|Australia|American Samoa|        31|         0|FIFA World Cup qu...|       Australia|       31|
|1971-09-13|   Tahiti|  Cook Islands|        30|         0| South Pacific Games|French Polynesia|       30|
|1979-08-30|     Fiji|      Kiribati|        24|         0| South Pacific Games|            Fiji|       24|
|2001-04-09|Australia|         Tonga|        22|         0|FIFA World Cup qu...|       Australia|       22|
|1966-04-03|    Libya|          Oman|        21|         0|            Arab Cup|            Iraq|       21|
+----------+---------+--------------+----------+----------+--------------------+----------------+---------+



**Cellule 30 — Q16**

In [86]:
spark.sql("""
SELECT
  team,
  SUM(goals_for) AS gf,
  SUM(goals_against) AS ga,
  SUM(goals_for) - SUM(goals_against) AS goal_average
FROM team_matches
GROUP BY team
ORDER BY goal_average DESC
""").show()

+-----------+----+----+------------+
|       team|  gf|  ga|goal_average|
+-----------+----+----+------------+
|     Brazil|2228| 910|        1318|
|    England|2282|1014|        1268|
|    Germany|2205|1133|        1072|
|  Argentina|1880|1042|         838|
|      Spain|1464| 650|         814|
|South Korea|1576| 808|         768|
|     Sweden|2060|1347|         713|
|Netherlands|1694|1000|         694|
|      Italy|1441| 803|         638|
|     Mexico|1592| 976|         616|
|       Iran| 968| 412|         556|
|     Russia|1229| 696|         533|
|    Hungary|1948|1429|         519|
|  Australia|1081| 583|         498|
|     France|1560|1128|         432|
|   China PR|1097| 674|         423|
|      Egypt|1096| 680|         416|
|      Japan|1189| 785|         404|
|   Scotland|1395| 997|         398|
|     Zambia|1199| 803|         396|
+-----------+----+----+------------+
only showing top 20 rows


**Cellule 31 — Q17**

In [87]:
spark.sql("""
WITH wins AS (
  SELECT year, team, COUNT(*) AS wins
  FROM team_results
  WHERE result='W'
  GROUP BY year, team
)
SELECT *
FROM (
  SELECT year, team, wins,
         ROW_NUMBER() OVER (PARTITION BY year ORDER BY wins DESC) AS rn
  FROM wins
)
WHERE rn <= 10
ORDER BY year DESC, rn
""").show()

+----+-------------+----+---+
|year|         team|wins| rn|
+----+-------------+----+---+
|2022|  South Korea|   9|  1|
|2022|   Costa Rica|   9|  2|
|2022|      Bahrain|   9|  3|
|2022|   Uzbekistan|   8|  4|
|2022|    Argentina|   8|  5|
|2022|      Senegal|   8|  6|
|2022|      Algeria|   8|  7|
|2022|      Tunisia|   8|  8|
|2022|       Mexico|   8|  9|
|2022|      Morocco|   7| 10|
|2021|United States|  17|  1|
|2021|      England|  15|  2|
|2021|      Denmark|  13|  3|
|2021|      Algeria|  13|  4|
|2021|       Canada|  13|  5|
|2021|       Mexico|  12|  6|
|2021|      Tunisia|  12|  7|
|2021|        Italy|  12|  8|
|2021|       Brazil|  12|  9|
|2021|      Morocco|  12| 10|
+----+-------------+----+---+
only showing top 20 rows


**Cellule 32 — Q18**

In [88]:
spark.sql("""
SELECT
  CAST(floor(year(date)/10)*10 AS INT) AS decade,
  COUNT(*) AS matches
FROM matches
GROUP BY CAST(floor(year(date)/10)*10 AS INT)
ORDER BY decade
""").show()

+------+-------+
|decade|matches|
+------+-------+
|  1870|     13|
|  1880|     55|
|  1890|     59|
|  1900|    124|
|  1910|    283|
|  1920|    729|
|  1930|   1010|
|  1940|    804|
|  1950|   1576|
|  1960|   2810|
|  1970|   3922|
|  1980|   4795|
|  1990|   6596|
|  2000|   9422|
|  2010|   9735|
|  2020|   2219|
+------+-------+



**Cellule 33 — Q19**

In [89]:
YEAR_TARGET = 2022

spark.sql(f"""
WITH per_team AS (
  SELECT year, team,
         SUM(CASE WHEN result='L' THEN 1 ELSE 0 END) AS losses,
         COUNT(*) AS played
  FROM team_results
  GROUP BY year, team
)
SELECT year, team, played
FROM per_team
WHERE year = {YEAR_TARGET} AND losses = 0 AND played > 0
ORDER BY played DESC
""").show()

+----+-------------+------+
|year|         team|played|
+----+-------------+------+
|2022|    Argentina|    12|
|2022|       Brazil|    11|
|2022|  Netherlands|    11|
|2022|      Georgia|     8|
|2022|French Guiana|     4|
|2022|       Angola|     4|
|2022|      Bonaire|     4|
|2022|    Palestine|     3|
|2022|  Puerto Rico|     2|
|2022|    Maule Sur|     2|
|2022|      Namibia|     1|
+----+-------------+------+



**Cellule 34 — Q20**

In [90]:
spark.sql("""
WITH ordered AS (
  SELECT
    team, date, result,
    CASE WHEN result='W' THEN 1 ELSE 0 END AS is_win,
    LAG(CASE WHEN result='W' THEN 1 ELSE 0 END) OVER (PARTITION BY team ORDER BY date) AS prev_is_win
  FROM team_results
),
breaks AS (
  SELECT *,
    CASE
      WHEN is_win=1 AND (prev_is_win IS NULL OR prev_is_win=0) THEN 1
      ELSE 0
    END AS new_streak
  FROM ordered
),
streak_id AS (
  SELECT *,
    SUM(new_streak) OVER (PARTITION BY team ORDER BY date) AS streak_group
  FROM breaks
),
streak_len AS (
  SELECT team, streak_group, COUNT(*) AS streak_wins
  FROM streak_id
  WHERE is_win=1
  GROUP BY team, streak_group
)
SELECT team, MAX(streak_wins) AS longest_win_streak
FROM streak_len
GROUP BY team
ORDER BY longest_win_streak DESC
""").show()

+--------------------+------------------+
|                team|longest_win_streak|
+--------------------+------------------+
|           Mauritius|                17|
|             Padania|                15|
|               Spain|                15|
|              France|                14|
|              Brazil|                14|
|              Guyana|                13|
|               Italy|                13|
|           Australia|                13|
|              Mexico|                13|
|            Scotland|                13|
|           German DR|                12|
|             Belgium|                12|
|             Germany|                12|
|           Indonesia|                12|
|             Morocco|                12|
|              Russia|                12|
|        Saudi Arabia|                11|
|              Sweden|                11|
|United Arab Emirates|                11|
|       United States|                11|
+--------------------+------------

**Cellule 35 — Q21**

In [91]:
spark.sql("""
WITH wins AS (
  SELECT tournament, team, COUNT(*) AS wins
  FROM (
    SELECT tournament, home_team AS team
    FROM matches
    WHERE home_score > away_score
    UNION ALL
    SELECT tournament, away_team AS team
    FROM matches
    WHERE away_score > home_score
  )
  GROUP BY tournament, team
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY tournament ORDER BY wins DESC) AS rn
  FROM wins
)
SELECT tournament, team, wins
FROM ranked
WHERE rn = 1
ORDER BY wins DESC
""").show()

+--------------------+-------------------+----+
|          tournament|               team|wins|
+--------------------+-------------------+----+
|            Friendly|            Germany| 307|
|British Championship|            England| 151|
|        Copa América|          Argentina| 127|
|FIFA World Cup qu...|             Mexico|  98|
|          CECAFA Cup|             Uganda|  92|
|UEFA Euro qualifi...|              Spain|  89|
| Nordic Championship|             Sweden|  83|
|  Merdeka Tournament|           Malaysia|  80|
|      FIFA World Cup|             Brazil|  73|
|African Cup of Na...|        Ivory Coast|  67|
|            Gold Cup|      United States|  67|
|African Cup of Na...|              Egypt|  60|
|            Gulf Cup|             Kuwait|  58|
|          King's Cup|           Thailand|  57|
|           Korea Cup|        South Korea|  51|
|   CFU Caribbean Cup|Trinidad and Tobago|  49|
|    AFF Championship|           Thailand|  42|
|       AFC Asian Cup|               Ira

**Cellule 36 — Q22**

In [92]:
spark.sql("""
SELECT
  team,
  SUM(CASE WHEN is_home=true  AND result='W' THEN 1 ELSE 0 END) AS home_wins,
  SUM(CASE WHEN is_home=true  AND result='D' THEN 1 ELSE 0 END) AS home_draws,
  SUM(CASE WHEN is_home=true  AND result='L' THEN 1 ELSE 0 END) AS home_losses,
  SUM(CASE WHEN is_home=false AND result='W' THEN 1 ELSE 0 END) AS away_wins,
  SUM(CASE WHEN is_home=false AND result='D' THEN 1 ELSE 0 END) AS away_draws,
  SUM(CASE WHEN is_home=false AND result='L' THEN 1 ELSE 0 END) AS away_losses
FROM team_results
GROUP BY team
ORDER BY (home_wins + away_wins) DESC
""").show()

+-----------+---------+----------+-----------+---------+----------+-----------+
|       team|home_wins|home_draws|home_losses|away_wins|away_draws|away_losses|
+-----------+---------+----------+-----------+---------+----------+-----------+
|     Brazil|      423|       112|         58|      228|        96|        102|
|    England|      324|       116|         83|      270|       137|        117|
|    Germany|      327|       113|         86|      246|        95|        119|
|  Argentina|      373|       126|         68|      174|       128|        145|
|     Sweden|      295|       106|        104|      222|       121|        204|
|South Korea|      296|       120|         83|      181|       115|        109|
|     Mexico|      320|       125|        104|      147|        91|        139|
|    Hungary|      265|       105|        104|      187|       106|        197|
|      Italy|      291|       123|         52|      154|       112|        105|
|     France|      290|       104|      